# Threshold Tuning, Fairness, and Language Equity

This notebook loads anonymized events from `data/ingested_events.jsonl`, computes basic group metrics, visualizes score distributions, and explores threshold sweeps.

In [13]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4)
data_path = Path('/workspace/data/ingested_events.jsonl')
print('Exists:', data_path.exists())

Exists: True


In [14]:
rows = []
if data_path.exists():
    with data_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                continue
            anon = event.get('anon_record', {})
            det = event.get('detector_result', {})
            rows.append({
                'language': anon.get('language') or 'unknown',
                'domain': anon.get('domain') or 'unknown',
                'source': anon.get('source') or 'unknown',
                'event_type': anon.get('event_type') or 'unknown',
                'risk_score': det.get('risk_score', 0.0),
                'label': det.get('label', 'unknown'),
                'action': det.get('action', 'allow')
            })
df = pd.DataFrame(rows)
df.head()

""


In [15]:
if df.empty:
    print('No ingested events yet. Ingest data first using the collector API.')
else:
    summary = df.groupby('language').agg(
        n_events=('risk_score', 'size'),
        mean_risk=('risk_score', 'mean'),
        escalate_rate=('action', lambda s: np.mean(s == 'escalate'))
    ).reset_index()
    print(summary)

No ingested events yet. Ingest data first using the collector API.


In [16]:
if not df.empty:
    for lang in df['language'].dropna().unique():
        subset = df[df['language'] == lang]
        plt.figure()
        plt.hist(subset['risk_score'], bins=10)
        plt.title(f'Risk Score Distribution - {lang}')
        plt.xlabel('risk_score')
        plt.ylabel('count')
        plt.show()

In [17]:
def simulate_actions(df_in, mid, high):
    out = df_in.copy()
    scores = out['risk_score'].values
    out['sim_action'] = np.where(scores >= high, 'escalate', np.where(scores >= mid, 'queue_for_review', 'allow'))
    return out

if not df.empty:
    gap_rows = []
    for mid, high in [(0.3, 0.6), (0.4, 0.7), (0.5, 0.8)]:
        sim = simulate_actions(df, mid, high)
        metrics = sim.groupby('language').agg(escalate_rate=('sim_action', lambda s: np.mean(s == 'escalate'))).reset_index()
        if not metrics.empty:
            gap_rows.append({
                'mid_threshold': mid,
                'high_threshold': high,
                'gap': float(metrics['escalate_rate'].max() - metrics['escalate_rate'].min())
            })
    print(pd.DataFrame(gap_rows).sort_values('gap'))